# AI-Powered Indian Tax Filing Assistant

## Project Overview
An intelligent tax advisor built using OpenAI's GPT-4o and Assistants API, designed to help users navigate Indian Income Tax Return (ITR) filing. The system includes document redaction for privacy, file upload capabilities, and personalized tax guidance.

## Key Features
- **Document Redaction**: Automatically redact sensitive information from tax documents
- **AI Tax Advisor**: Custom-trained assistant with Indian tax law knowledge
- **File Processing**: Upload and analyze Form 26AS, AIS, bank statements
- **Interactive Guidance**: Step-by-step tax filing assistance

## Tech Stack
- **Python**: Core programming language
- **OpenAI API**: GPT-4o and Assistants API
- **PyMuPDF (fitz)**: PDF processing and redaction
- **python-dotenv**: Environment variable management

## Disclaimer
This is a demonstration project for educational purposes. Always consult qualified tax professionals for actual tax filing.

## 1. Environment Setup

In [1]:
# Install required packages
# Run this cell only once or when setting up the environment

import subprocess
import sys

def install_package(package):
    """Install a package using pip"""
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Required packages
packages = [
    "openai",
    "pymupdf", 
    "python-dotenv",
]

# Install packages (uncomment if needed)
for package in packages:
    install_package(package)

print("Environment setup complete!")

Environment setup complete!


In [2]:
# Import required libraries
import openai
import fitz  # PyMuPDF
import os
import time
import re
from dotenv import load_dotenv

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. User Credentials & Configuration

In [4]:
# Load environment variables from .env file
# Create a .env file in your project directory with:
# OPENAI_API_KEY=your_openai_api_key_here

load_dotenv("api_key.env")

# Set OpenAI API key from environment variable
openai.api_key = os.getenv("OPENAI_API_KEY")

if not openai.api_key:
    raise ValueError("OpenAI API key not found. Please set OPENAI_API_KEY in your .env file.")

print("✅ API credentials loaded successfully!")

✅ API credentials loaded successfully!


In [ ]:
# Configuration: File paths and sensitive data for redaction
# Note: Replace these with your actual file paths and sensitive information

# Sample file paths (replace with your actual paths)
FILE_PATHS = {
    "ATS": "path/to/your/ATS_document.pdf",
    "AIS": "path/to/your/AIS_document.pdf", 
    "BANK_HDFC": "path/to/your/HDFC_statement.pdf",
    "BANK_SBI": "path/to/your/SBI_statement.pdf"
}

# Sensitive information to redact (replace with your actual sensitive data)
SENSITIVE_DATA = {
    "ATS": ["XXXXX1234X", "JOHN DOE", "123 SAMPLE ADDRESS", "CITY, STATE, 123456"],
    "AIS": ["XXXXX1234X", "9876543210", "sample@email.com"],
    "BANK_GENERAL": ["JOHN DOE", "SAMPLE ADDRESS", "ACCOUNT_NUMBER_PLACEHOLDER"]
}

print("📁 File paths and redaction data configured!")

## 3. PDF Redaction Module

In [ ]:
def redact_pdf(input_path, output_path, sensitive_words):
    """
    Redact sensitive information from a PDF document.
    
    Args:
        input_path (str): Path to the input PDF file
        output_path (str): Path where redacted PDF will be saved
        sensitive_words (list): List of sensitive words/phrases to redact
    
    Returns:
        bool: True if successful, False otherwise
    """
    try:
        # Open the PDF document
        doc = fitz.open(input_path)
        
        # Process each page
        for page in doc:
            # Search and redact each sensitive word
            for word in sensitive_words:
                text_instances = page.search_for(word)
                for inst in text_instances:
                    # Add black redaction annotation
                    page.add_redact_annot(inst, fill=(0, 0, 0))
            
            # Apply all redactions on this page
            page.apply_redactions()
        
        # Save the redacted document
        doc.save(output_path)
        doc.close()
        
        print(f"✅ Successfully redacted and saved: {output_path}")
        return True
        
    except Exception as e:
        print(f"❌ Error during PDF redaction: {str(e)}")
        return False

def batch_redact_documents(file_paths, sensitive_data):
    """
    Redact multiple documents in batch.
    
    Args:
        file_paths (dict): Dictionary of document types and their paths
        sensitive_data (dict): Dictionary of document types and their sensitive data
    """
    for doc_type, path in file_paths.items():
        if os.path.exists(path):
            output_path = f"redacted_{doc_type.lower()}.pdf"
            words_to_redact = sensitive_data.get(doc_type, [])
            redact_pdf(path, output_path, words_to_redact)
        else:
            print(f"⚠️  File not found: {path}")

print("📄 PDF redaction functions loaded!")

## 4. Tax Assistant Creation

In [ ]:
# Define the Tax Assistant system prompt
# This can be personalized by changing USER_NAME

USER_NAME = "User"  # Change to "Suraj" for personalized experience

TAX_ASSISTANT_PROMPT = f"""
You are {USER_NAME}'s dedicated Tax Maestro: a sharp, no-nonsense Indian tax expert and compliance strategist. Your primary goal is to guide {USER_NAME} through ITR filing with maximum efficiency and minimal mental load.

**Core Directives:**
1. **Diagnose & Strategize:** Ask targeted questions to understand income, deductions, and documents (Form 16, 26AS, AIS, bank statements). Proactively suggest legal deductions and filing strategies.
2. **Reconcile & Mitigate:** Identify and guide resolution for document mismatches. Advise on conservative declarations, explanatory notes, and necessary documentation to mitigate compliance risks.
3. **Action-Oriented Communication:** Prioritize concrete next steps. Use checklists, bullet points, and direct commands. Avoid jargon. If {USER_NAME} is overwhelmed, provide immediate, clear actions. If energetic, leverage it for progress.
4. **Verify, Never Assume:** Always confirm facts. Guide defensive filing even if no tax is owed.
5. **Outcome:** Ensure accurate, compliant ITR filing, not just understanding. You are an executor and shield.
"""

print("📝 Tax Assistant prompt configured!")

In [ ]:
def create_tax_assistant():
    """
    Create an OpenAI Assistant for tax guidance.
    
    Returns:
        Assistant object or None if creation fails
    """
    try:
        assistant = openai.beta.assistants.create(
            name="Tax Maestro",
            instructions=TAX_ASSISTANT_PROMPT,
            model="gpt-4o",
            tools=[{"type": "file_search"}],
        )
        
        print(f"✅ Tax Assistant created successfully! ID: {assistant.id}")
        return assistant
        
    except Exception as e:
        print(f"❌ Error creating assistant: {str(e)}")
        return None

def create_conversation_thread():
    """
    Create a new conversation thread.
    
    Returns:
        Thread object or None if creation fails
    """
    try:
        thread = openai.beta.threads.create()
        print(f"✅ Conversation thread created! ID: {thread.id}")
        return thread
        
    except Exception as e:
        print(f"❌ Error creating thread: {str(e)}")
        return None

print("🤖 Assistant creation functions loaded!")

## 5. File Upload & Processing

In [ ]:
def upload_file_to_openai(file_path):
    """
    Upload a file to OpenAI for use with Assistants API.
    
    Args:
        file_path (str): Path to the file to upload
    
    Returns:
        str: File ID if successful, None otherwise
    """
    try:
        if not os.path.exists(file_path):
            print(f"❌ File not found: {file_path}")
            return None
            
        with open(file_path, "rb") as file:
            upload_response = openai.files.create(
                file=file,
                purpose="assistants"
            )
        
        print(f"✅ File uploaded successfully! ID: {upload_response.id}")
        return upload_response.id
        
    except Exception as e:
        print(f"❌ Error uploading file: {str(e)}")
        return None

def send_message_with_file(thread_id, message_content, file_id=None):
    """
    Send a message to the conversation thread, optionally with a file attachment.
    
    Args:
        thread_id (str): Thread ID for the conversation
        message_content (str): The message to send
        file_id (str, optional): File ID to attach to the message
    
    Returns:
        Message object or None if sending fails
    """
    try:
        message_params = {
            "thread_id": thread_id,
            "role": "user",
            "content": message_content
        }
        
        # Add file attachment if provided
        if file_id:
            message_params["attachments"] = [
                {
                    "file_id": file_id,
                    "tools": [{"type": "file_search"}]
                }
            ]
        
        message = openai.beta.threads.messages.create(**message_params)
        print(f"✅ Message sent successfully!")
        return message
        
    except Exception as e:
        print(f"❌ Error sending message: {str(e)}")
        return None

print("📤 File upload functions loaded!")

## 6. AI Assistant Run Routine

In [ ]:
def run_assistant_and_get_response(assistant_id, thread_id, max_wait_time=60):
    """
    Run the assistant and wait for completion, then retrieve the response.
    
    Args:
        assistant_id (str): ID of the assistant to run
        thread_id (str): ID of the thread to run on
        max_wait_time (int): Maximum time to wait for completion (seconds)
    
    Returns:
        str: Assistant's response or None if failed
    """
    try:
        # Start the assistant run
        run = openai.beta.threads.runs.create(
            assistant_id=assistant_id,
            thread_id=thread_id
        )
        
        print(f"🔄 Assistant run started... ID: {run.id}")
        
        # Poll for completion with timeout
        elapsed_time = 0
        while elapsed_time < max_wait_time:
            run_status = openai.beta.threads.runs.retrieve(
                thread_id=thread_id, 
                run_id=run.id
            )
            
            if run_status.status == "completed":
                print("✅ Assistant run completed!")
                break
            elif run_status.status == "failed":
                print(f"❌ Assistant run failed: {run_status.last_error}")
                return None
                
            time.sleep(2)
            elapsed_time += 2
            
        if elapsed_time >= max_wait_time:
            print(f"⏰ Assistant run timed out after {max_wait_time} seconds")
            return None
        
        # Retrieve the latest assistant message
        messages = openai.beta.threads.messages.list(thread_id=thread_id)
        
        for message in messages:
            if message.role == "assistant":
                return message.content[0].text.value
                
        return None
        
    except Exception as e:
        print(f"❌ Error running assistant: {str(e)}")
        return None

print("🏃 Assistant run functions loaded!")

## 7. Error Handling & Utilities

In [ ]:
def validate_environment():
    """
    Validate that all required environment variables and dependencies are set up.
    
    Returns:
        bool: True if environment is valid, False otherwise
    """
    # Check API key
    if not openai.api_key:
        print("❌ OpenAI API key not configured")
        return False
    
    # Test API connection
    try:
        openai.models.list()
        print("✅ OpenAI API connection successful")
    except Exception as e:
        print(f"❌ OpenAI API connection failed: {str(e)}")
        return False
    
    return True

def markdown_to_ansi(text):
    """
    Convert markdown formatting to ANSI escape codes for terminal display.
    
    Args:
        text (str): Text with markdown formatting
    
    Returns:
        str: Text with ANSI formatting
    """
    # Bold text
    text = re.sub(r'\*\*(.*?)\*\*', r'\033[1m\1\033[0m', text)
    # Italic text
    text = re.sub(r'\*(.*?)\*', r'\033[3m\1\033[0m', text)
    # Inline code
    text = re.sub(r'`(.*?)`', r'\033[96m\1\033[0m', text)
    return text

def display_formatted_response(response_text):
    """
    Display assistant response with proper formatting.
    
    Args:
        response_text (str): Raw response text from assistant
    """
    if response_text:
        formatted = markdown_to_ansi(response_text)
        print("\n" + "="*50)
        print("🤖 TAX MAESTRO RESPONSE:")
        print("="*50)
        print(formatted)
        print("="*50 + "\n")
    else:
        print("❌ No response received from assistant")

print("🛠️ Error handling and utility functions loaded!")

## 8. User Interface Functions

In [ ]:
class TaxAssistantUI:
    """
    Simple command-line interface for the Tax Assistant.
    """
    
    def __init__(self):
        self.assistant = None
        self.thread = None
        
    def initialize(self):
        """
        Initialize the assistant and thread.
        
        Returns:
            bool: True if initialization successful
        """
        print("🚀 Initializing Tax Assistant...")
        
        # Validate environment
        if not validate_environment():
            return False
        
        # Create assistant and thread
        self.assistant = create_tax_assistant()
        self.thread = create_conversation_thread()
        
        if self.assistant and self.thread:
            print("✅ Tax Assistant ready for use!")
            return True
        else:
            print("❌ Failed to initialize Tax Assistant")
            return False
    
    def ask_question(self, question, file_path=None):
        """
        Ask the tax assistant a question, optionally with a file.
        
        Args:
            question (str): Question to ask
            file_path (str, optional): Path to file to upload with question
        """
        if not self.assistant or not self.thread:
            print("❌ Assistant not initialized. Call initialize() first.")
            return
        
        file_id = None
        
        # Upload file if provided
        if file_path:
            file_id = upload_file_to_openai(file_path)
            if not file_id:
                print("❌ File upload failed. Proceeding without file.")
        
        # Send message
        message = send_message_with_file(self.thread.id, question, file_id)
        if not message:
            return
        
        # Get response
        response = run_assistant_and_get_response(self.assistant.id, self.thread.id)
        display_formatted_response(response)
    
    def show_menu(self):
        """
        Display the main menu options.
        """
        print("\n📋 TAX ASSISTANT MENU:")
        print("1. Ask a tax question")
        print("2. Upload and analyze document")
        print("3. Get ITR filing checklist")
        print("4. Exit")
        print("-" * 30)

print("🖥️ User interface functions loaded!")

## 9. Main Application Runner

In [ ]:
def main():
    """
    Main application entry point.
    """
    print("🏛️ Welcome to the AI Tax Filing Assistant!")
    print("This tool helps you navigate Indian Income Tax Return filing.\n")
    
    # Initialize the UI
    tax_ui = TaxAssistantUI()
    
    if not tax_ui.initialize():
        print("❌ Failed to initialize. Please check your setup.")
        return
    
    # Example usage - you can uncomment and modify these
    
    # Example 1: Simple question
    # tax_ui.ask_question("What documents do I need for ITR filing?")
    
    # Example 2: Question with file upload
    # tax_ui.ask_question(
    #     "Please analyze my Form 26AS and suggest next steps.",
    #     file_path="path/to/your/form26as.pdf"
    # )
    
    print("\n🎯 Tax Assistant is ready! Use the methods above to interact.")
    print("\n📚 Example usage:")
    print("tax_ui.ask_question('What ITR form should I use?')")
    print("tax_ui.ask_question('Analyze my documents', 'path/to/document.pdf')")

# Run the main application
if __name__ == "__main__":
    main()

## 10. Example Usage & Testing

In [ ]:
# Example: Initialize and use the Tax Assistant
# Uncomment the lines below to test the system

# Step 1: Create the UI instance
# tax_assistant = TaxAssistantUI()

# Step 2: Initialize (creates assistant and thread)
# if tax_assistant.initialize():
#     print("Ready to use!")

# Step 3: Ask questions
# tax_assistant.ask_question("What are the key documents I need for ITR filing?")

# Step 4: Upload and analyze a document
# tax_assistant.ask_question(
#     "Please analyze this Form 26AS and tell me what to do next.",
#     file_path="redacted_form26as.pdf"
# )

print("📖 Example usage code ready! Uncomment lines above to test.")

## 11. Document Redaction Demo

In [ ]:
# Example: Redact sensitive information from tax documents
# Uncomment and modify the code below to redact your documents

# Example sensitive data (replace with your actual sensitive information)
DEMO_SENSITIVE_WORDS = [
    "XXXXX1234X",  # PAN number placeholder
    "JOHN DOE",     # Name placeholder
    "9876543210",  # Phone placeholder
    "sample@email.com",  # Email placeholder
    "123 SAMPLE ADDRESS"  # Address placeholder
]

# Example redaction (uncomment to use)
# input_pdf = "path/to/your/original_document.pdf"
# output_pdf = "redacted_document.pdf"
# 
# if redact_pdf(input_pdf, output_pdf, DEMO_SENSITIVE_WORDS):
#     print(f"Document successfully redacted and saved as {output_pdf}")
# else:
#     print("Redaction failed. Please check file paths and permissions.")

print("🔒 Document redaction demo ready! Uncomment lines above to test.")

## Project Notes & Next Steps

### Current Capabilities
- ✅ PDF document redaction for privacy
- ✅ Custom AI assistant with Indian tax expertise
- ✅ File upload and analysis workflow
- ✅ Interactive Q&A with tax documents
- ✅ Formatted response display

### Potential Improvements
- 🔄 **Web UI**: Create a Streamlit or Flask web interface
- 🔄 **Database Integration**: Store conversation history and user preferences
- 🔄 **Multi-document Analysis**: Compare multiple tax documents simultaneously
- 🔄 **Export Features**: Generate ITR summary reports
- 🔄 **Validation Rules**: Add Indian tax law validation checks

### Setup Instructions
1. Create a `.env` file with your OpenAI API key:
   ```
   OPENAI_API_KEY=your_openai_api_key_here
   ```
2. Install requirements:
   ```bash
   pip install openai pymupdf python-dotenv
   ```
3. Replace placeholder file paths and sensitive data with your actual information
4. Run the notebook cells in order

### Disclaimer
This is a demonstration project for educational and portfolio purposes. Always consult qualified tax professionals for actual tax filing decisions.